# Continual adapter objective — compact walkthrough

This notebook isolates the public engineering contract behind target-adapter updates. It demonstrates the loss components and bounded replay; it is not a full DINOv3 training run.

In [1]:
%pip install -q git+https://github.com/EfeErim/bitirmeprojesi.git@aads-public-demo-v1.1.1

In [2]:
import numpy as np

from aads_public.training import LossWeights, ReplayBuffer, continual_loss

loss = continual_loss(
    logits=np.array([[3.0, 0.2], [0.1, 2.5]]),
    labels=np.array([0, 1]),
    current_features=np.array([[1.0, 0.0], [0.0, 1.0]]),
    teacher_features=np.array([[0.9, 0.1], [0.1, 0.9]]),
    adapter_parameters=np.array([0.2, -0.2]),
    weights=LossWeights(classification=1.0, distillation=0.5, adapter_l2=0.1),
)
replay = ReplayBuffer(capacity=4, seed=7)
for sample_id in range(20):
    replay.add(sample_id)

print(f"classification={loss.classification:.4f}")
print(f"distillation={loss.distillation:.4f}")
print(f"adapter_l2={loss.adapter_l2:.4f}")
print(f"total={loss.total:.4f}")
print(f"bounded_replay_size={len(replay)}")

classification=0.0729
distillation=0.0100
adapter_l2=0.0400
total=0.0819
bounded_replay_size=4


The maintained private workflow adds actual DINOv3/PEFT modules, held-out ID evaluation, locked OOD evidence, checkpoint recovery, and promotion gates. This notebook deliberately stops at the dependency-light mathematical contract.